# GNSS Positioning from First Principles
**Javier Esteban Porras López · Industrial Engineer → GNSS Systems**

---

This notebook implements GPS/Galileo positioning from scratch:
orbit propagation (SGP4), pseudorange modelling, Weighted Least Squares
receiver solution, Klobuchar ionospheric correction, and DOP analysis.

**Libraries required:**
```
pip install sgp4 numpy matplotlib requests
```

**What we'll build:**
1. Fetch live GPS TLEs → propagate satellite positions (ECI → ECEF → geodetic)
2. Simulate pseudoranges with realistic error sources
3. Solve receiver position via Weighted Least Squares (iterate to convergence)
4. Apply Klobuchar ionospheric correction
5. Compute PDOP / GDOP / HDOP / VDOP from satellite geometry
6. Build an error budget


---
## 1 · Imports and Physical Constants


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec
import requests
from datetime import datetime, timezone, timedelta
import math
import warnings
warnings.filterwarnings('ignore')

# SGP4 propagator  (pip install sgp4)
from sgp4.api import Satrec, jday

# ── Physical constants ──────────────────────────────────────────────────
C        = 299_792.458       # speed of light  (km/s)
MU       = 398_600.4418      # Earth's gravitational parameter  (km³/s²)
OMEGA_E  = 7.2921151467e-5   # Earth's rotation rate  (rad/s)

# WGS-84 ellipsoid
A_EARTH  = 6_378.137         # semi-major axis  (km)
F_EARTH  = 1 / 298.257223563 # flattening
B_EARTH  = A_EARTH * (1 - F_EARTH)  # semi-minor axis  (km)
E2_EARTH = 2*F_EARTH - F_EARTH**2   # first eccentricity squared

# Mask angle — ignore satellites below this elevation
MASK_DEG = 5.0               # degrees

print(f'Speed of light: {C:,.3f} km/s')
print(f'Earth semi-major axis (WGS-84): {A_EARTH:.3f} km')
print(f'Mask angle: {MASK_DEG}°')


---
## 2 · Coordinate System Transformations

GNSS positioning uses three coordinate systems:

| Frame | Origin | X axis | Rotates with Earth? | Used for |
|-------|--------|--------|---------------------|----------|
| **ECI** | Earth centre | Vernal equinox | ✗ | SGP4 output |
| **ECEF** | Earth centre | Greenwich meridian | ✓ | Receiver coordinates |
| **Geodetic** | Observer | — | ✓ | Everyday lat/lon/alt |
| **ENU** | Observer | East | ✓ | Elevation / azimuth |

**ECI → ECEF:** rotate by GMST (Greenwich Mean Sidereal Time), the angle
Earth has rotated from the vernal equinox:

$$
\mathbf{r}_{\text{ECEF}} =
\begin{bmatrix} \cos\theta_G & \sin\theta_G & 0 \\
               -\sin\theta_G & \cos\theta_G & 0 \\
                0 & 0 & 1 \end{bmatrix}
\mathbf{r}_{\text{ECI}}
$$

**ECEF → Geodetic:** iterative (Bowring's method) to account for the
ellipsoidal shape of Earth — the radius varies from 6356 km (poles)
to 6378 km (equator).


In [ ]:
def gmst_rad(jd_full):
    """
    Greenwich Mean Sidereal Time (radians) from Julian Date.
    Formula: IAU 1982 GMST model.
    """
    T = (jd_full - 2_451_545.0) / 36_525.0   # Julian centuries from J2000
    theta_deg = (280.46061837
                 + 360.98564736629 * (jd_full - 2_451_545.0)
                 + 0.000387933 * T**2
                 - T**3 / 38_710_000.0)
    return np.radians(theta_deg % 360)


def eci_to_ecef(r_eci, theta_gmst):
    """
    Rotate position vector from ECI to ECEF.
    r_eci      : (3,) or (N,3)  position in ECI  [km]
    theta_gmst : GMST angle  [rad]
    """
    c, s = np.cos(theta_gmst), np.sin(theta_gmst)
    R = np.array([[ c,  s, 0],
                  [-s,  c, 0],
                  [ 0,  0, 1]])
    return (R @ r_eci.T).T if r_eci.ndim == 2 else R @ r_eci


def ecef_to_geodetic(x, y, z):
    """
    ECEF (km) → geodetic (lat°, lon°, alt km).
    Uses Bowring's iterative method for the latitude.
    """
    a, b = A_EARTH, B_EARTH
    ep2 = (a**2 - b**2) / b**2   # second eccentricity squared

    p   = np.sqrt(x**2 + y**2)   # distance from Z axis
    lon = np.arctan2(y, x)

    # Bowring's method — iterate until latitude converges
    theta = np.arctan2(z * a, p * b)
    lat   = np.arctan2(z + ep2 * b * np.sin(theta)**3,
                       p -  E2_EARTH * a * np.cos(theta)**3)
    for _ in range(5):            # 5 iterations is more than enough
        N   = a / np.sqrt(1 - E2_EARTH * np.sin(lat)**2)  # radius of curvature
        lat = np.arctan2(z + E2_EARTH * N * np.sin(lat), p)

    N   = a / np.sqrt(1 - E2_EARTH * np.sin(lat)**2)
    alt = p / np.cos(lat) - N    # altitude above ellipsoid  [km]

    return np.degrees(lat), np.degrees(lon), alt


def geodetic_to_ecef(lat_deg, lon_deg, alt_km):
    """
    Geodetic (lat°, lon°, alt km) → ECEF (km).
    Closed-form (no iteration needed in this direction).
    """
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)
    N = A_EARTH / np.sqrt(1 - E2_EARTH * np.sin(lat)**2)
    x = (N + alt_km) * np.cos(lat) * np.cos(lon)
    y = (N + alt_km) * np.cos(lat) * np.sin(lon)
    z = (N * (1 - E2_EARTH) + alt_km) * np.sin(lat)
    return np.array([x, y, z])


def ecef_to_enu(r_ecef_sat, r_ecef_obs, lat_deg, lon_deg):
    """
    Vector from observer to satellite in East-North-Up frame.
    This is the local 'sky' coordinate system from your location.
    """
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)
    sl, cl = np.sin(lat), np.cos(lat)
    sn, cn = np.sin(lon), np.cos(lon)
    # Rotation matrix ECEF → ENU
    R = np.array([[-sn,     cn,    0  ],
                  [-sl*cn, -sl*sn,  cl ],
                  [ cl*cn,  cl*sn,  sl ]])
    dr = r_ecef_sat - r_ecef_obs   # vector from obs to sat
    return R @ dr


def elevation_azimuth(enu):
    """
    Elevation (deg, +upward) and azimuth (deg, 0=N clockwise) from ENU vector.
    """
    e, n, u = enu
    rng = np.linalg.norm(enu)
    el  = np.degrees(np.arcsin(u / rng))
    az  = np.degrees(np.arctan2(e, n)) % 360
    return el, az, rng


# Quick sanity check
r_obs = geodetic_to_ecef(48.137, 11.576, 0.0)  # Munich
lat_c, lon_c, alt_c = ecef_to_geodetic(*r_obs)
print(f'Round-trip test — Munich: {lat_c:.4f}°N, {lon_c:.4f}°E, {alt_c*1000:.1f} m')


---
## 3 · TLE Data and SGP4 Propagation

A **TLE (Two-Line Element set)** encodes a satellite's orbit in 6 Keplerian
elements plus correction terms for perturbations:

```
GPS BIIR-2  (PRN 13)
1 24876U 97035A   25178.79542245  .00000019  00000+0  00000+0 0  9993
2 24876  55.2847  72.1234 0055432  60.4001 300.0123  2.00563224210987
         ──────  ───────  ───────  ───────  ────────  ──────────
         incl°   RAAN°   ecc      ω°       M°        n (rev/day)
```

| Field | Symbol | GPS (typical) | Galileo (typical) |
|-------|--------|---------------|-------------------|
| Inclination | $i$ | 55° | 56° |
| Eccentricity | $e$ | ≈ 0.005 | ≈ 0.001 |
| Mean motion | $n$ | 2.006 rev/day | 1.706 rev/day |
| Semi-major axis | $a$ | ≈ 26,560 km | ≈ 29,601 km |

**SGP4** (Simplified General Perturbations 4) integrates the orbit analytically,
accounting for Earth's $J_2$ oblateness, drag, lunar/solar gravity, and
solar radiation pressure. Accuracy: < 1 km for TLEs < 3 days old.


In [ ]:
def fetch_tle(constellation='GPS'):
    """
    Fetch TLEs from CelesTrak for GPS or Galileo.
    Returns list of (name, line1, line2) tuples.
    CelesTrak mirrors USSPACECOM's catalog, updated multiple times/day.
    """
    url = f'https://celestrak.org/GNSS/table.php?constellation={constellation}&format=TLE'
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        lines = [l.strip() for l in r.text.strip().split('\n') if l.strip()]
    except Exception as e:
        print(f'  ⚠ Could not fetch live data ({e}). Using fallback demo TLEs.')
        # 3-satellite demo subset (PRN 1, 7, 13) — always works offline
        lines = [
            'GPS BIIR-2  (PRN 13)',
            '1 24876U 97035A   25178.79542245  .00000019  00000+0  00000+0 0  9999',
            '2 24876  55.2847  72.1234 0055432  60.4001 300.0123  2.00563224210987',
            'GPS BIIR-10 (PRN 07)',
            '1 26360U 00025A   25178.91234567  .00000023  00000+0  00000+0 0  9998',
            '2 26360  55.0912 132.4560 0110892 245.6789  112.345  2.00562820197654',
            'GPS BIIRM-1 (PRN 17)',
            '1 28874U 05038A   25178.55432167  .00000021  00000+0  00000+0 0  9997',
            '2 28874  55.3012 252.7890 0143210  78.9012 282.3456  2.00562619187412',
        ]

    sats = []
    for i in range(0, len(lines) - 2, 3):
        name, l1, l2 = lines[i], lines[i+1], lines[i+2]
        if l1.startswith('1 ') and l2.startswith('2 '):
            sats.append((name.replace('0 ', '').strip(), l1, l2))
    return sats


def build_satrecs(tle_list):
    """
    Parse TLE list into sgp4 Satrec objects.
    Satrec stores the Keplerian elements + perturbation coefficients.
    """
    records = []
    for name, l1, l2 in tle_list:
        try:
            sat = Satrec.twoline2rv(l1, l2)
            records.append({'name': name, 'satrec': sat, 'l2': l2})
        except Exception:
            pass
    return records


print('Fetching GPS TLEs from CelesTrak…')
gps_tles   = fetch_tle('GPS')
gps_satrec = build_satrecs(gps_tles)
print(f'  GPS: {len(gps_satrec)} satellites parsed')

print('Fetching Galileo TLEs from CelesTrak…')
gal_tles   = fetch_tle('Galileo')
gal_satrec = build_satrecs(gal_tles)
print(f'  Galileo: {len(gal_satrec)} satellites parsed')

all_satrec = gps_satrec + gal_satrec
print(f'\nTotal: {len(all_satrec)} GNSS satellites loaded')


In [ ]:
def propagate_all(satrec_list, epoch):
    """
    Propagate all satellites to a given UTC datetime.
    Returns list of dicts with ECI, ECEF, geodetic positions.

    SGP4 output is in ECI (km).  We then:
      1. Compute GMST for the epoch
      2. Rotate ECI → ECEF (Earth-fixed)
      3. Convert ECEF → geodetic (lat/lon/alt)
    """
    jd, fr = jday(epoch.year, epoch.month, epoch.day,
                  epoch.hour, epoch.minute, epoch.second + epoch.microsecond/1e6)
    theta  = gmst_rad(jd + fr)

    results = []
    for rec in satrec_list:
        e_code, r_eci, v_eci = rec['satrec'].sgp4(jd, fr)
        if e_code != 0:      # 0 = success, non-zero = propagation error
            continue
        r_eci  = np.array(r_eci)   # km, ECI
        r_ecef = eci_to_ecef(r_eci, theta)
        lat, lon, alt = ecef_to_geodetic(*r_ecef)
        results.append({
            'name'  : rec['name'],
            'r_eci' : r_eci,
            'r_ecef': r_ecef,
            'lat'   : lat,
            'lon'   : lon,
            'alt'   : alt,
            'const' : 'GPS' if 'GPS' in rec['name'] or 'PRN' in rec['name'] else 'Galileo'
        })
    return results, theta


# Propagate to now
NOW    = datetime.now(timezone.utc)
posns, theta_now = propagate_all(all_satrec, NOW)

print(f'Propagated {len(posns)} satellites to {NOW.strftime("%Y-%m-%d %H:%M:%S UTC")}')

# Show a sample
print(f'\n{"Satellite":<32} {"Lat":>8} {"Lon":>9} {"Alt (km)":>10}')
print('-' * 62)
for p in posns[:6]:
    print(f"{p['name']:<32} {p['lat']:>7.2f}° {p['lon']:>8.2f}° {p['alt']:>10.1f}")


---
## 4 · Ground Track Visualisation

The **ground track** is the satellite's sub-satellite point over time —
the point directly below it on the Earth's surface.

GPS ground tracks repeat every **sidereal day** (≈23h 56min) because
GPS orbital period is exactly half a sidereal day by design.
This means a GPS satellite passes over the same ground point every day
— 4 minutes earlier in solar time.

Galileo ground tracks repeat every **10 sidereal days** (period is
10/17 of a sidereal day, so the pattern takes 10 days to close).


In [ ]:
def compute_ground_tracks(satrec_list, epoch, duration_min=120, dt_min=2):
    """
    Compute ground tracks over duration_min minutes, sampled every dt_min minutes.
    Returns {satellite_name: [(lon, lat), ...]}
    """
    times  = [epoch + timedelta(minutes=t) for t in range(0, duration_min, dt_min)]
    tracks = {rec['name']: [] for rec in satrec_list}

    for t in times:
        jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute,
                      t.second + t.microsecond/1e6)
        theta  = gmst_rad(jd + fr)
        for rec in satrec_list:
            e_code, r_eci, _ = rec['satrec'].sgp4(jd, fr)
            if e_code != 0:
                continue
            r_ecef = eci_to_ecef(np.array(r_eci), theta)
            lat, lon, _ = ecef_to_geodetic(*r_ecef)
            tracks[rec['name']].append((lon, lat))

    return tracks


# Build ground tracks for first 6 GPS + first 4 Galileo sats
subset = gps_satrec[:6] + gal_satrec[:4]
tracks = compute_ground_tracks(subset, NOW, duration_min=200, dt_min=3)

# ── Plot ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#060e1c')
ax.set_facecolor('#060e1c')

# Grid
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
for lo in range(-180, 181, 30):
    ax.axvline(lo, color='#1a2e4a', linewidth=0.5, zorder=0)
for la in range(-90, 91, 30):
    lw = 0.9 if la == 0 else 0.4
    ax.axhline(la, color='#1a2e4a', linewidth=lw, zorder=0)

# Tropics
for la in [23.44, -23.44]:
    ax.axhline(la, color='#2a3a20', linewidth=0.6, linestyle='--', zorder=0)

# Ground tracks
for rec in subset:
    name = rec['name']
    pts  = tracks.get(name, [])
    if not pts:
        continue
    is_gps = ('GPS' in name or 'PRN' in name)
    col    = '#3fd2c7' if is_gps else '#a78bfa'
    lons   = [p[0] for p in pts]
    lats   = [p[1] for p in pts]

    # Split track at antimeridian (±180°) to avoid ugly wrap-around lines
    segs = []
    seg  = []
    for i, (lo, la) in enumerate(zip(lons, lats)):
        if seg and abs(lo - seg[-1][0]) > 180:
            segs.append(seg); seg = []
        seg.append((lo, la))
    if seg:
        segs.append(seg)

    for s in segs:
        sx = [p[0] for p in s]
        sy = [p[1] for p in s]
        ax.plot(sx, sy, color=col, linewidth=0.9, alpha=0.6, zorder=2)

# Current positions
for p in posns:
    if p['name'] not in tracks:
        continue
    is_gps = p['const'] == 'GPS'
    col    = '#3fd2c7' if is_gps else '#a78bfa'
    ax.plot(p['lon'], p['lat'], 'o', color=col, markersize=5, zorder=4,
            markeredgecolor='white', markeredgewidth=0.4)

# Labels
ax.set_xlabel('Longitude (°)', color='#6a8098', fontsize=10)
ax.set_ylabel('Latitude (°)',  color='#6a8098', fontsize=10)
ax.tick_params(colors='#4a6070')
for sp in ax.spines.values():
    sp.set_color('#1a2e4a')

gps_patch = plt.Line2D([0],[0], color='#3fd2c7', lw=2, label=f'GPS (6 shown)')
gal_patch = plt.Line2D([0],[0], color='#a78bfa', lw=2, label=f'Galileo (4 shown)')
ax.legend(handles=[gps_patch, gal_patch], loc='lower left',
          framealpha=0.3, labelcolor='#cfe0f0', fontsize=9,
          facecolor='#0b1f3a', edgecolor='#1a2e4a')

ax.set_title(f'GNSS Ground Tracks · Next 3h 20min from {NOW.strftime("%H:%M UTC")}',
             color='#c0d8e8', pad=10, fontsize=11)

plt.tight_layout()
plt.savefig('ground_tracks.png', dpi=150, bbox_inches='tight', facecolor='#060e1c')
plt.show()
print('Figure saved as ground_tracks.png')


---
## 5 · Pseudorange Model

Each GNSS satellite continuously broadcasts:
- Its precise position (from the **ephemeris**, broadcast every 30 s)
- A precise timestamp (from an onboard **atomic clock**, accurate to ≈20 ns)

Your receiver measures the signal arrival time. The difference is the
**travel time**, and multiplying by $c$ gives a *pseudorange* $\rho_i$:

$$
\boxed{
\rho_i = \underbrace{r_i}_{\text{true range}} +
         \underbrace{c \cdot \delta t_r}_{\text{receiver clock}} -
         \underbrace{c \cdot \delta t_s}_{\text{sat clock}} +
         \underbrace{I_i}_{\text{ionosphere}} +
         \underbrace{T_i}_{\text{troposphere}} +
         \underbrace{\varepsilon_i}_{\text{noise}}
}
$$

**Why 4 satellites?**  
We have 4 unknowns: $x, y, z$ (position) + $\delta t_r$ (receiver clock error).  
Each satellite gives one equation → need $N \geq 4$.

**Error magnitudes:**

| Source | Magnitude | Mitigated by |
|--------|-----------|-------------|
| Receiver clock | up to 300 km (1 ms) | Solved as 4th unknown |
| Ionosphere ($I_i$) | 1–15 m (L1) | Klobuchar model / dual-freq |
| Troposphere ($T_i$) | 2–25 m | Saastamoinen model |
| Satellite clock | < 1 m | Broadcast corrections |
| Multipath | 0–5 m | Choke-ring antennas |
| Noise ($\varepsilon_i$) | 0.3–1 m | Averaging, filtering |


In [ ]:
# Observer: Stuttgart, Germany
OBS_LAT = 48.775
OBS_LON =  9.182
OBS_ALT =  0.247   # km

r_obs_ecef = geodetic_to_ecef(OBS_LAT, OBS_LON, OBS_ALT)


def compute_visibility(posns, r_obs_ecef, lat, lon, mask_deg=5.0):
    """
    For each satellite, compute elevation, azimuth, range from observer.
    Keep only satellites above mask angle.
    """
    visible = []
    for p in posns:
        enu      = ecef_to_enu(p['r_ecef'], r_obs_ecef, lat, lon)
        el, az, rng = elevation_azimuth(enu)
        if el >= mask_deg:
            visible.append({**p, 'elevation': el, 'azimuth': az, 'range': rng})
    return visible


# Noise model: user-equivalent range error as function of elevation
# Satellites at low elevation have longer signal path through atmosphere.
# We model σ(el) = σ_zenith / sin(el) — the simple "sin(el)" weighting.
SIGMA_ZENITH = 0.5   # m  (zenith UERE)

def uere(elevation_deg):
    """User Equivalent Range Error (m) as function of elevation angle."""
    return SIGMA_ZENITH / math.sin(math.radians(max(elevation_deg, 5)))


def simulate_pseudoranges(visible_sats, clock_error_m=0.0, seed=42):
    """
    Simulate pseudoranges for each visible satellite.

    rho_i = range_i * 1000   +  clock_bias (m)  +  noise (m)

    range is in km from propagation; we convert to metres here.
    clock_error_m = c * delta_t_receiver (metres equivalent).
    """
    rng = np.random.default_rng(seed)
    pseudoranges = []

    for s in visible_sats:
        true_range_m = s['range'] * 1000.0   # km → m
        sigma        = uere(s['elevation'])   # noise std dev in metres
        noise        = rng.normal(0, sigma)   # simulated random noise

        rho = true_range_m + clock_error_m + noise
        pseudoranges.append({
            **s,
            'true_range_m': true_range_m,
            'noise_m'     : noise,
            'rho_m'       : rho,
            'sigma_m'     : sigma,
            'weight'      : 1.0 / sigma**2    # WLS weight = 1/sigma²
        })

    return pseudoranges


# Visible satellites from Stuttgart right now
visible = compute_visibility(posns, r_obs_ecef, OBS_LAT, OBS_LON)
CLOCK_ERROR_M = 1500.0   # simulate 5 μs receiver clock error (= 1.5 km)
pseudo         = simulate_pseudoranges(visible, clock_error_m=CLOCK_ERROR_M)

print(f'Observer: Stuttgart ({OBS_LAT}°N, {OBS_LON}°E)')
print(f'Visible satellites (>{MASK_DEG}°): {len(visible)}')
print(f'Simulated receiver clock error: {CLOCK_ERROR_M/1000:.3f} km = {CLOCK_ERROR_M/C/1000*1e6:.2f} μs\n')

print(f'{"Satellite":<28} {"Const":<8} {"El (°)":>7} {"True rng (km)":>14} {"Pseudorng (km)":>15} {"σ (m)":>7}')
print('-' * 84)
for p in sorted(pseudo, key=lambda x: -x['elevation']):
    print(f"{p['name']:<28} {p['const']:<8} {p['elevation']:>6.1f}°  "
          f"{p['true_range_m']/1000:>13.1f}  {p['rho_m']/1000:>14.1f}  {p['sigma_m']:>6.2f}")


---
## 6 · Weighted Least Squares Receiver Solution

We have $N$ pseudorange equations and 4 unknowns: $\mathbf{u} = [x, y, z, b]^T$
where $b = c \cdot \delta t_r$ is the receiver clock bias in metres.

The geometric range to satellite $i$ from an estimated position $\hat{\mathbf{r}}$ is:

$$r_i(\hat{\mathbf{r}}) = \|\mathbf{r}_{s,i} - \hat{\mathbf{r}}\|$$

**Linearise** the pseudorange residual $\Delta \rho_i = \rho_i - r_i(\hat{\mathbf{r}}) - \hat{b}$:

$$\Delta \rho_i \approx -\hat{\mathbf{e}}_i^T \Delta\mathbf{r} + \Delta b$$

where $\hat{\mathbf{e}}_i = (\hat{\mathbf{r}} - \mathbf{r}_{s,i})/r_i$ is the unit vector *from* satellite to receiver.

Stacking all equations: $\mathbf{H} \Delta\mathbf{x} = \Delta\boldsymbol{\rho}$

**Design matrix H:**
$$H_{i} = [-e_{x,i},\ -e_{y,i},\ -e_{z,i},\ 1]$$

**Weighted Least Squares solution** (weight = $1/\sigma_i^2$):
$$\Delta\mathbf{x} = (\mathbf{H}^T \mathbf{W} \mathbf{H})^{-1} \mathbf{H}^T \mathbf{W} \Delta\boldsymbol{\rho}$$

Iterate until $|\Delta\mathbf{x}|$ is negligible (typically 3–5 iterations).


In [ ]:
def wls_position(pseudo_list, initial_ecef=None, max_iter=10, tol_m=0.001):
    """
    Weighted Least Squares GNSS positioning.

    Parameters
    ----------
    pseudo_list  : list of pseudorange dicts (from simulate_pseudoranges)
    initial_ecef : starting ECEF position (km); defaults to Earth's centre
    max_iter     : iteration limit
    tol_m        : convergence tolerance in metres

    Returns
    -------
    dict with final ECEF position, clock bias, residuals, covariance,
    number of iterations, and convergence flag.
    """
    if len(pseudo_list) < 4:
        raise ValueError(f'Need ≥4 pseudoranges; got {len(pseudo_list)}')

    # Initialise at Earth's centre (or provided estimate)
    x = np.array(initial_ecef if initial_ecef is not None else [0.0, 0.0, 0.0])
    b = 0.0         # clock bias (metres)

    # Satellite positions in metres
    r_sats = np.array([p['r_ecef'] * 1000.0 for p in pseudo_list])  # km → m
    rho    = np.array([p['rho_m']           for p in pseudo_list])
    W_diag = np.array([p['weight']          for p in pseudo_list])
    W      = np.diag(W_diag)

    history = []    # track convergence

    for iteration in range(max_iter):
        # Geometric ranges from current estimate
        dr     = r_sats - x * 1000.0        # vectors sat → receiver (m)
        ranges = np.linalg.norm(dr, axis=1) # scalar ranges (m)

        # Unit vectors FROM satellite TO receiver
        e_hat  = -dr / ranges[:, np.newaxis]

        # Design matrix H  (N × 4)
        H = np.column_stack([e_hat, np.ones(len(pseudo_list))])

        # Pseudorange residuals
        delta_rho = rho - (ranges + b)

        # WLS update:  Δx = (HᵀWH)⁻¹ HᵀW Δρ
        HtW     = H.T @ W
        HtWH    = HtW @ H
        delta_x = np.linalg.solve(HtWH, HtW @ delta_rho)

        dx   = delta_x[:3]   # position update (m)
        db   = delta_x[3]    # clock bias update (m)

        x   += dx / 1000.0   # keep x in km
        b   += db

        step_m = np.linalg.norm(dx)
        history.append(step_m)

        if step_m < tol_m:
            break

    # Covariance matrix (geometry only, not scaled by noise)
    Q = np.linalg.inv(H.T @ W @ H)

    return {
        'ecef_km'   : x,
        'clock_bias': b,
        'residuals' : delta_rho,
        'Q'         : Q,
        'H'         : H,
        'n_iter'    : iteration + 1,
        'converged' : step_m < tol_m,
        'history'   : history
    }


# ── Solve ────────────────────────────────────────────────────────────────
sol = wls_position(pseudo)

# Convert solution to geodetic
sol_lat, sol_lon, sol_alt = ecef_to_geodetic(*sol['ecef_km'])

# 3D position error (compare to known observer location)
r_sol   = sol['ecef_km'] * 1000     # m
r_true  = r_obs_ecef     * 1000     # m
error_3d = np.linalg.norm(r_sol - r_true)

print('═' * 55)
print('  WEIGHTED LEAST SQUARES SOLUTION')
print('═' * 55)
print(f'  Iterations:       {sol["n_iter"]}  (converged: {sol["converged"]})')
print(f'  Solved position:  {sol_lat:.4f}°N,  {sol_lon:.4f}°E')
print(f'  True position:    {OBS_LAT:.4f}°N,  {OBS_LON:.4f}°E')
print(f'  Altitude:         {sol_alt*1000:.1f} m')
print(f'  3D position error: {error_3d:.1f} m')
print(f'  Clock bias solved: {sol["clock_bias"]:.2f} m  (true: {CLOCK_ERROR_M:.2f} m)')
print(f'  Clock error:       {abs(sol["clock_bias"] - CLOCK_ERROR_M):.2f} m residual')

# Convergence plot
fig, ax = plt.subplots(figsize=(7, 3.5))
fig.patch.set_facecolor('#060e1c'); ax.set_facecolor('#060e1c')
ax.semilogy(range(1, len(sol['history'])+1), sol['history'],
            'o-', color='#3fd2c7', markersize=6, linewidth=1.8)
ax.axhline(0.001, color='#f2c14e', linestyle='--', linewidth=0.9, label='1 mm tolerance')
ax.set_xlabel('Iteration', color='#6a8098'); ax.set_ylabel('Step size (m)', color='#6a8098')
ax.set_title('WLS Convergence', color='#c0d8e8', pad=8)
ax.tick_params(colors='#4a6070'); ax.legend(labelcolor='#9fb4cc', facecolor='#0b1f3a', edgecolor='#1a2e4a')
for sp in ax.spines.values(): sp.set_color('#1a2e4a')
plt.tight_layout()
plt.savefig('wls_convergence.png', dpi=150, bbox_inches='tight', facecolor='#060e1c')
plt.show()


---
## 7 · Ionospheric Correction: Klobuchar Model

The **ionosphere** (50–1000 km altitude) is a layer of free electrons
that slows GNSS signals. The delay depends on the **Total Electron Content
(TEC)** along the signal path, varying with:
- Solar activity (day/night, 11-year cycle)
- Latitude (geomagnetic equator anomaly)
- Elevation angle (longer path at low elevation)

**Klobuchar (1987) model** — the single-frequency GPS broadcast correction.  
It removes ≈50% of the ionospheric delay on average.  
(Dual-frequency receivers cancel ionosphere entirely: $f_1$ vs $f_2$.)

The GPS navigation message broadcasts 8 coefficients: $\alpha_0 \ldots \alpha_3$
and $\beta_0 \ldots \beta_3$.  We use GPS-ICD nominal values here.

**Model steps:**
1. Map receiver location to ionospheric pierce point (IPP) at 350 km altitude
2. Compute local solar time at IPP
3. Interpolate a cosine wave scaled by $\alpha$ coefficients
4. Apply obliquity factor $F$ (longer path at low elevation)

$$I = F \cdot \left[ 5 \times 10^{-9} + A_p \cos\left(\frac{2\pi(t - 50400)}{P_r}\right) \right] \cdot c$$


In [ ]:
def klobuchar_correction(obs_lat_deg, obs_lon_deg, elevation_deg, azimuth_deg,
                         gps_tow_sec, alpha, beta):
    """
    Klobuchar single-frequency ionospheric correction (GPS-ICD IS-200J).

    Parameters
    ----------
    obs_lat_deg   : observer latitude  (degrees)
    obs_lon_deg   : observer longitude (degrees)
    elevation_deg : satellite elevation (degrees)
    azimuth_deg   : satellite azimuth  (degrees)
    gps_tow_sec   : GPS time of week   (seconds)
    alpha         : [a0, a1, a2, a3]  broadcast iono amplitude coefficients
    beta          : [b0, b1, b2, b3]  broadcast iono period   coefficients

    Returns
    -------
    iono_delay_m : ionospheric delay in metres (L1 frequency)
    """
    el_rad = math.radians(max(elevation_deg, 0))
    az_rad = math.radians(azimuth_deg)

    phi_u  = math.radians(obs_lat_deg)  # observer latitude in radians
    lam_u  = math.radians(obs_lon_deg)  # observer longitude in radians

    # 1. Earth-centred angle (semi-circles)
    psi = 0.0137 / (el_rad / math.pi + 0.11) - 0.022

    # 2. Ionospheric pierce-point latitude  (semi-circles)
    phi_i = phi_u / math.pi + psi * math.cos(az_rad)
    phi_i = max(-0.416, min(0.416, phi_i))

    # 3. IPP longitude (semi-circles)
    lam_i = lam_u / math.pi + psi * math.sin(az_rad) / math.cos(phi_i * math.pi)

    # 4. Geomagnetic latitude of IPP (semi-circles)
    phi_m = phi_i + 0.064 * math.cos((lam_i - 1.617) * math.pi)

    # 5. Local solar time at IPP (seconds)
    t_sec = (43200.0 * lam_i + gps_tow_sec) % 86400

    # 6. Amplitude and period from broadcast coefficients
    A_p = sum(alpha[n] * phi_m**n for n in range(4))
    A_p = max(0.0, A_p)               # non-negative

    P_r = sum(beta[n]  * phi_m**n for n in range(4))
    P_r = max(72000.0, P_r)           # period ≥ 20 hours

    # 7. Phase (radians)
    X = 2 * math.pi * (t_sec - 50400.0) / P_r

    # 8. Slant factor (obliquity): larger for low-elevation signals
    F = 1.0 + 16.0 * (0.53 - el_rad / math.pi)**3

    # 9. Vertical delay
    if abs(X) < 1.57:
        T_iono = 5e-9 + A_p * (1 - X**2/2 + X**4/24)   # Taylor expansion of cos
    else:
        T_iono = 5e-9

    # 10. Slant delay in metres
    iono_delay_m = F * T_iono * C * 1000  # C is in km/s → ×1000 for m
    return iono_delay_m


# GPS nominal broadcast iono coefficients (moderate solar activity)
ALPHA = [2.0e-8, 1.490e-8, -5.960e-8, -5.960e-8]
BETA  = [8.192e4, 0.0,     -6.553e4,   5.243e4]

# GPS time of week for current time
GPS_EPOCH = datetime(1980, 1, 6, tzinfo=timezone.utc)
gps_tow   = (NOW - GPS_EPOCH).total_seconds() % (7 * 86400)

# Apply Klobuchar correction to all visible satellites
pseudo_corrected = []
for p in pseudo:
    iono_m = klobuchar_correction(
        OBS_LAT, OBS_LON, p['elevation'], p['azimuth'],
        gps_tow, ALPHA, BETA
    )
    rho_corrected = p['rho_m'] - iono_m    # subtract ionospheric delay
    pseudo_corrected.append({**p, 'iono_m': iono_m, 'rho_corrected': rho_corrected})

# Show effect of correction
print(f'{"Satellite":<28} {"El (°)":>6} {"Iono delay (m)":>14} {"Corr rng (km)":>14}')
print('-' * 68)
for p in sorted(pseudo_corrected, key=lambda x: -x['elevation']):
    print(f"{p['name']:<28} {p['elevation']:>5.1f}°  {p['iono_m']:>13.2f}  {p['rho_corrected']/1000:>13.2f}")

# Plot ionospheric delay vs elevation angle
el_range = np.linspace(5, 90, 200)
iono_values = [klobuchar_correction(OBS_LAT, OBS_LON, el, 180, gps_tow, ALPHA, BETA)
               for el in el_range]

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor('#060e1c'); ax.set_facecolor('#060e1c')
ax.plot(el_range, iono_values, color='#f2c14e', linewidth=2)
for p in pseudo_corrected:
    ax.scatter(p['elevation'], p['iono_m'], color='#3fd2c7', s=30, zorder=5)
ax.set_xlabel('Elevation angle (°)', color='#6a8098')
ax.set_ylabel('Ionospheric delay (m)', color='#6a8098')
ax.set_title('Klobuchar Model — Ionospheric Delay vs Elevation', color='#c0d8e8', pad=8)
ax.tick_params(colors='#4a6070')
for sp in ax.spines.values(): sp.set_color('#1a2e4a')
plt.tight_layout()
plt.savefig('iono_correction.png', dpi=150, bbox_inches='tight', facecolor='#060e1c')
plt.show()


In [ ]:
# Compare positioning: uncorrected vs Klobuchar-corrected pseudoranges

# Build corrected pseudorange list
pseudo_iono = [{**p, 'rho_m': p['rho_corrected']} for p in pseudo_corrected]

sol_raw  = wls_position(pseudo,      )
sol_iono = wls_position(pseudo_iono  )

lat_raw,  lon_raw,  _ = ecef_to_geodetic(*sol_raw['ecef_km'])
lat_iono, lon_iono, _ = ecef_to_geodetic(*sol_iono['ecef_km'])

err_raw  = np.linalg.norm(sol_raw['ecef_km']  * 1000 - r_obs_ecef * 1000)
err_iono = np.linalg.norm(sol_iono['ecef_km'] * 1000 - r_obs_ecef * 1000)

print('Positioning with simulated noise + 5 μs clock error:')
print(f'  Uncorrected:            {err_raw:.1f} m 3D error')
print(f'  Klobuchar-corrected:    {err_iono:.1f} m 3D error')
print(f'  Improvement:            {err_raw - err_iono:.1f} m')
print(f'  (Real GPS L1 accuracy ≈ 3–5 m with full corrections)')


---
## 8 · Dilution of Precision (DOP)

**DOP** quantifies how satellite geometry amplifies ranging errors into position errors:

$$\text{Position error} = \text{DOP} \times \text{UERE}$$

Computed from the covariance matrix $\mathbf{Q} = (\mathbf{H}^T\mathbf{H})^{-1}$
(with uniform weighting), transformed to **East-North-Up** coordinates:

| DOP | Formula | What it measures |
|-----|---------|------------------|
| GDOP | $\sqrt{\text{tr}(Q)}$ | Overall geometry (3D + time) |
| PDOP | $\sqrt{Q_{xx}+Q_{yy}+Q_{zz}}$ | 3D position |
| HDOP | $\sqrt{Q_{ee}+Q_{nn}}$ | Horizontal (E + N) |
| VDOP | $\sqrt{Q_{uu}}$ | Vertical (Up) |
| TDOP | $\sqrt{Q_{tt}}$ | Time |

Interpretation:
- **PDOP < 2**: Excellent — typical in open sky
- **PDOP 2–5**: Good
- **PDOP 5–10**: Moderate — urban canyons, obstructions
- **PDOP > 10**: Poor — avoid critical applications


In [ ]:
def compute_dop(visible_sats, obs_lat, obs_lon):
    """
    Compute GDOP, PDOP, HDOP, VDOP, TDOP in ENU coordinates.

    Design matrix rows: [e, n, u, 1]  (ENU unit vectors + clock column)
    This gives DOP in the local East-North-Up frame.
    """
    if len(visible_sats) < 4:
        return {k: float('inf') for k in ['GDOP','PDOP','HDOP','VDOP','TDOP']}

    rows = []
    for s in visible_sats:
        el  = math.radians(s['elevation'])
        az  = math.radians(s['azimuth'])
        e   = math.cos(el) * math.sin(az)   # East component
        n   = math.cos(el) * math.cos(az)   # North component
        u   = math.sin(el)                  # Up component
        rows.append([e, n, u, 1.0])

    H  = np.array(rows)
    try:
        Q  = np.linalg.inv(H.T @ H)
    except np.linalg.LinAlgError:
        return {k: float('inf') for k in ['GDOP','PDOP','HDOP','VDOP','TDOP']}

    return {
        'GDOP': math.sqrt(Q[0,0]+Q[1,1]+Q[2,2]+Q[3,3]),  # all 4 unknowns
        'PDOP': math.sqrt(Q[0,0]+Q[1,1]+Q[2,2]),          # position only
        'HDOP': math.sqrt(Q[0,0]+Q[1,1]),                  # horizontal
        'VDOP': math.sqrt(Q[2,2]),                         # vertical
        'TDOP': math.sqrt(Q[3,3]),                         # time
        'Q'   : Q
    }


dop = compute_dop(visible, OBS_LAT, OBS_LON)

print(f'DOP values for {OBS_LAT}°N, {OBS_LON}°E  ({len(visible)} visible sats):')
print(f'  GDOP: {dop["GDOP"]:.2f}  (overall)')
print(f'  PDOP: {dop["PDOP"]:.2f}  (3D position)')
print(f'  HDOP: {dop["HDOP"]:.2f}  (horizontal)')
print(f'  VDOP: {dop["VDOP"]:.2f}  (vertical   — always worse than HDOP)')
print(f'  TDOP: {dop["TDOP"]:.2f}  (time)')

qual = 'Excellent' if dop['PDOP'] < 2 else ('Good' if dop['PDOP'] < 5 else 'Moderate')
print(f'  Geometry quality: {qual}')


# ── Sky plot + DOP breakdown ────────────────────────────────────────────
fig = plt.figure(figsize=(13, 5))
fig.patch.set_facecolor('#060e1c')
gs  = GridSpec(1, 2, figure=fig, width_ratios=[1, 1.1], wspace=0.35)

# Left: sky plot
ax_sky = fig.add_subplot(gs[0], projection='polar')
ax_sky.set_facecolor('#060e1c')
ax_sky.set_theta_zero_location('N')   # North at top
ax_sky.set_theta_direction(-1)        # azimuth increases clockwise
ax_sky.set_ylim(0, 90)
ax_sky.set_yticks([0, 30, 60, 90])
ax_sky.set_yticklabels(['90°','60°','30°','0°'], color='#4a6070', fontsize=7)
ax_sky.tick_params(colors='#3a5060', labelsize=7)
ax_sky.grid(color='#1a2e4a', linewidth=0.6)
ax_sky.set_facecolor('#040c18')

# Mask angle ring
theta_ring = np.linspace(0, 2*np.pi, 200)
ax_sky.plot(theta_ring, np.full_like(theta_ring, 90 - MASK_DEG),
            '--', color='#f2c14e', linewidth=0.8, alpha=0.5)

for s in visible:
    az_r  = math.radians(s['azimuth'])
    el_r  = 90 - s['elevation']    # in polar: 0 = center (zenith), 90 = edge (horizon)
    col   = '#3fd2c7' if s['const'] == 'GPS' else '#a78bfa'
    ax_sky.scatter(az_r, el_r, color=col, s=60, zorder=5,
                   edgecolors='white', linewidths=0.4)
    ax_sky.text(az_r + 0.08, el_r + 3, s['name'].split()[-1],
                color=col, fontsize=6.5, ha='left', va='bottom')

ax_sky.set_title('Sky Plot', color='#c0d8e8', pad=10, fontsize=10)

# Right: DOP bar chart
ax_dop = fig.add_subplot(gs[1])
ax_dop.set_facecolor('#060e1c')

dop_names  = ['GDOP', 'PDOP', 'HDOP', 'VDOP', 'TDOP']
dop_vals   = [dop[k] for k in dop_names]
colors_dop = ['#3fd2c7' if v < 2 else ('#f2c14e' if v < 5 else '#e06c6c') for v in dop_vals]

bars = ax_dop.barh(dop_names, dop_vals, color=colors_dop, height=0.55,
                   edgecolor='rgba(255,255,255,0.1)')

for bar, val in zip(bars, dop_vals):
    ax_dop.text(val + 0.05, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', color='#c0d8e8', fontsize=9)

ax_dop.axvline(2, color='#3fd2c7', linewidth=0.8, linestyle='--', alpha=0.5, label='Excellent')
ax_dop.axvline(5, color='#f2c14e', linewidth=0.8, linestyle='--', alpha=0.5, label='Good')
ax_dop.set_xlim(0, max(dop_vals) * 1.25)
ax_dop.set_xlabel('DOP value', color='#6a8098')
ax_dop.set_title(f'DOP — {len(visible)} visible sats', color='#c0d8e8', pad=8)
ax_dop.tick_params(colors='#4a6070')
ax_dop.legend(fontsize=8, labelcolor='#9fb4cc', facecolor='#0b1f3a', edgecolor='#1a2e4a')
for sp in ax_dop.spines.values(): sp.set_color('#1a2e4a')

plt.savefig('dop_analysis.png', dpi=150, bbox_inches='tight', facecolor='#060e1c')
plt.show()


---
## 9 · Error Budget

GPS **User Range Accuracy (URA)** = RSS (root-sum-square) of all error sources.

The final position error at 95% confidence:

$$\text{SEP}_{95} \approx 2 \times \text{UERE} \times \text{PDOP}$$

where **UERE** = √(Σσᵢ²) is the User Equivalent Range Error.

Modern GPS (with SBAS augmentation) achieves UERE ≈ 0.5–1 m.


In [ ]:
# GPS L1 C/A error budget — typical values for open-sky, good receiver
error_budget = [
    ('Satellite clock',       0.30,  'Mitigated by broadcast clock corrections'),
    ('Satellite orbit',       0.20,  'Residual ephemeris error after correction'),
    ('Ionosphere (Klobuchar)',2.00,  '~50% of raw delay removed; residual varies'),
    ('Troposphere',           0.70,  'Varies with weather; Saastamoinen model'),
    ('Receiver noise',        0.30,  'Tracking loop + thermal noise'),
    ('Multipath',             1.00,  'Reflections; worst in urban canyons'),
]

names  = [e[0]  for e in error_budget]
sigmas = [e[1]  for e in error_budget]
notes  = [e[2]  for e in error_budget]

uere_total = math.sqrt(sum(s**2 for s in sigmas))
sep_95     = 2 * uere_total * dop['PDOP']

print(f'GPS L1 C/A — Error Budget')
print('=' * 60)
print(f'{"Source":<28} {"σ (m)":>6}  {"σ² (m²)":>9}')
print('-' * 60)
for n, s in zip(names, sigmas):
    print(f'{n:<28} {s:>6.2f}  {s**2:>9.4f}')
print('-' * 60)
print(f'{"UERE (RSS)":<28} {uere_total:>6.2f}  {uere_total**2:>9.4f}')
print(f'PDOP: {dop["PDOP"]:.2f}')
print(f'SEP (95%): ≈ 2 × {uere_total:.2f} × {dop["PDOP"]:.2f} = {sep_95:.1f} m')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#060e1c')
for ax in axes: ax.set_facecolor('#060e1c')

# Horizontal bar chart of individual errors
palette = ['#3fd2c7','#3fd2c7','#f2c14e','#f2c14e','#3fd2c7','#e06c6c']
axes[0].barh(names, sigmas, color=palette, height=0.55, edgecolor='none')
for i, (s, n) in enumerate(zip(sigmas, names)):
    axes[0].text(s + 0.04, i, f'{s:.2f} m', va='center', color='#c0d8e8', fontsize=8.5)
axes[0].set_xlabel('σ (metres)', color='#6a8098')
axes[0].set_title('Individual Error Sources (1σ)', color='#c0d8e8', pad=8)
axes[0].tick_params(colors='#4a6070')
for sp in axes[0].spines.values(): sp.set_color('#1a2e4a')

# Pie chart of variance contribution
variances = [s**2 for s in sigmas]
explode   = [0.04]*len(variances)
wedge_col = palette
axes[1].pie(variances, labels=names, autopct='%1.0f%%', colors=wedge_col,
            startangle=90, pctdistance=0.75, explode=explode,
            textprops={'color':'#8aafcc','fontsize':8})
axes[1].set_title(f'Variance share  ·  UERE = {uere_total:.2f} m',
                  color='#c0d8e8', pad=8)

plt.suptitle(f'GPS L1 C/A Error Budget  ·  PDOP = {dop["PDOP"]:.2f}  ·  SEP(95%) ≈ {sep_95:.0f} m',
             color='#c0d8e8', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('error_budget.png', dpi=150, bbox_inches='tight', facecolor='#060e1c')
plt.show()


---
## Summary

In this notebook we implemented the full GPS single-point positioning chain from scratch:

| Step | What we did | Key concept |
|------|------------|-------------|
| 1 | Fetched live TLEs from CelesTrak | TLE = orbital description |
| 2 | Propagated with SGP4 | TLE + time → ECI position |
| 3 | ECI → ECEF → Geodetic | 3 coordinate systems |
| 4 | Simulated pseudoranges | ρ = r + c·δt + noise |
| 5 | Weighted Least Squares | 4 unknowns: x, y, z, δt |
| 6 | Klobuchar ionospheric correction | ~50% iono delay removed |
| 7 | DOP computation | Satellite geometry quality |
| 8 | Error budget | UERE × PDOP → position accuracy |

**Next steps:**
- Dual-frequency (L1/L5 or E1/E5a) to eliminate ionosphere completely
- Extended Kalman Filter for kinematic positioning
- Carrier-phase measurements for cm-level RTK accuracy
- Multi-constellation (GPS + Galileo) for better DOP

---
*Javier Esteban Porras López · [Portfolio](https://industrialengineerestebanlopez-tech.github.io) · 
GNSS Academy JSNP Master — Stuttgart, Germany*
